# Ecuación de ondas 1D: cuaderno guiado con verificaciones

Este cuaderno propone una implementación paso a paso del esquema explícito centrado para la ecuación de ondas en una dimensión.

Trabajaremos con:

$$
\frac{\partial^2 u}{\partial t^2}=c^2\frac{\partial^2u}{\partial x^2},\quad x\in(0,1),\ t\in(0,T)
$$

con fronteras homogéneas y datos iniciales:

$$u(0,t)=u(1,t)=0,\qquad u(x,0)=\sin(\pi x),\qquad u_t(x,0)=0.$$

`T` es el tiempo final de simulación (`t_final`).


## 1) Discretización del esquema centrado

Con malla uniforme $x_i=i\Delta x$ y $t^n=n\Delta t$:

$$
\frac{u_i^{n+1}-2u_i^n+u_i^{n-1}}{\Delta t^2}
=
 c^2\frac{u_{i+1}^n-2u_i^n+u_{i-1}^n}{\Delta x^2}.
$$

Despejando:

$$u_i^{n+1}=2u_i^n-u_i^{n-1}+\sigma^2\left(u_{i+1}^n-2u_i^n+u_{i-1}^n\right),
\quad \sigma=\frac{c\Delta t}{\Delta x}.$$

Para iniciar la recursión, usamos:

$$u_i^1=u_i^0+\Delta t\,v_i^0+\frac{\sigma^2}{2}\left(u_{i+1}^0-2u_i^0+u_{i-1}^0\right).$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Parámetros base
c = 1.0
L = 1.0
T = 0.6
N = 220
sigma_obj = 0.9


## 2) Funciones de apoyo (solución exacta y condiciones iniciales)


In [ ]:
def exact_solution(x, t, c=1.0):
    return np.sin(np.pi * x) * np.cos(c * np.pi * t)


def u0(x):
    return np.sin(np.pi * x)


def v0(x):
    return np.zeros_like(x)


## 3) TO DO A: construir malla y parámetros discretos


In [ ]:
def build_grid(L, T, N, sigma_obj, c=1.0):
    # TO DO
    # Debe retornar x, dx, dt, nt, sigma
    dx = L / N
    dt = sigma_obj * dx / c
    nt = max(1, int(np.ceil(T / dt)))
    dt = T / nt
    sigma = c * dt / dx
    x = np.linspace(0.0, L, N + 1)
    return x, dx, dt, nt, sigma


## 4) TO DO B: primer paso temporal y paso general


In [ ]:
def first_step(u_prev, v_init, sigma, dt):
    # TO DO
    u_curr = u_prev.copy()
    u_curr[1:-1] = (
        u_prev[1:-1]
        + dt * v_init[1:-1]
        + 0.5 * sigma**2 * (u_prev[2:] - 2.0 * u_prev[1:-1] + u_prev[:-2])
    )
    return u_curr


def wave_step(u_prev, u_curr, sigma):
    # TO DO
    u_next = np.empty_like(u_curr)
    u_next[1:-1] = (
        2.0 * u_curr[1:-1]
        - u_prev[1:-1]
        + sigma**2 * (u_curr[2:] - 2.0 * u_curr[1:-1] + u_curr[:-2])
    )
    return u_next


## 5) TO DO C: solver completo con historial


In [ ]:
def solve_wave(c=1.0, L=1.0, T=0.6, N=220, sigma_obj=0.9, store_every=5):
    x, dx, dt, nt, sigma = build_grid(L, T, N, sigma_obj, c=c)

    u_prev = u0(x)
    u_prev[0], u_prev[-1] = 0.0, 0.0

    v_init = v0(x)
    u_curr = first_step(u_prev, v_init, sigma, dt)
    u_curr[0], u_curr[-1] = 0.0, 0.0

    t_hist = [0.0, dt]
    U_hist = [u_prev.copy(), u_curr.copy()]

    for n in range(1, nt):
        u_next = wave_step(u_prev, u_curr, sigma)
        t_next = (n + 1) * dt
        u_next[0], u_next[-1] = 0.0, 0.0

        if (n + 1) % store_every == 0 or (n + 1) == nt:
            t_hist.append(t_next)
            U_hist.append(u_next.copy())

        u_prev, u_curr = u_curr, u_next

    t_final = nt * dt
    return x, u_curr, t_final, sigma, np.array(t_hist), np.array(U_hist)


## 6) Verificaciones automáticas

Validamos tres puntos:

- Condición CFL para el esquema explícito: $\sigma\leq 1$.
- Fronteras homogéneas conservadas.
- Error L2 frente a la solución exacta en `t_final`.


In [ ]:
x, u_num, t_final, sigma, t_hist, U_hist = solve_wave(c=c, L=L, T=T, N=N, sigma_obj=sigma_obj, store_every=5)
u_ex = exact_solution(x, t_final, c=c)

assert sigma <= 1.0 + 1e-12, f"Sigma fuera de rango: {sigma:.4f}"
assert abs(u_num[0]) < 1e-12 and abs(u_num[-1]) < 1e-12, "Fronteras no homogéneas"

err_l2 = np.sqrt(np.mean((u_num - u_ex) ** 2))

print(f"t_final={t_final:.6f}")
print(f"sigma={sigma:.6f}")
print(f"error L2={err_l2:.3e}")


## 7) Visualización: comparación en el tiempo final


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, u_num, 'o-', ms=3, lw=1.6, label='Numérica (DF centradas)')
plt.plot(x, u_ex, '--', lw=2, label='Exacta')
plt.title('Ecuación de ondas 1D en t_final')
plt.xlabel('x')
plt.ylabel('u(x,t_final)')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## 8) Visualización: evolución temporal en el cuaderno


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
line_num, = ax.plot([], [], lw=2, label='Numérica')
line_ex, = ax.plot([], [], '--', lw=2, label='Exacta')
ax.set_xlim(0, 1)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel('x')
ax.set_ylabel('u(x,t)')
ax.grid(alpha=0.25)
ax.legend()

for k in range(0, len(t_hist), max(1, len(t_hist)//12)):
    t = t_hist[k]
    line_num.set_data(x, U_hist[k])
    line_ex.set_data(x, exact_solution(x, t, c=c))
    ax.set_title(f'Evolución temporal t={t:.3f}')
    plt.pause(0.08)

plt.show()


## 9) Ejercicios propuestos

1. Cambia la condición inicial a $u(x,0)=\sin(2\pi x)$ y compara fase y amplitud numérica.
2. Evalúa el error para distintos $N$, manteniendo $\sigma$ fijo en 0.9.
3. Repite el experimento con $\sigma>1$ y documenta el patrón de inestabilidad.
